## Ejercicio Integrador: Sistema de Gestión de Entregas de TPs

Es viernes a la noche y se acaba de cerrar el deadline del TP1 de Introducción al Desarrollo de Software. Sos uno de los correctores de la materia y tu responsabilidad es organizar todas las entregas que llegaron antes de que el equipo docente empiece a corregir el lunes.

El problema: los alumnos entregan sus soluciones como archivos `.txt` en una carpeta compartida. Algunos archivos vienen con nombres crípticos (`tp_final_v3_FINAL.txt`, `entrega_juancito.txt`), y no hay garantía de que todos hayan respetado el formato pedido en el enunciado. Necesitás un script que te permita validar y organizar todas las entregas de forma automatizada, porque hacerlo a mano con más de 250 alumnos no es una opción.

Cada archivo de entrega debe tener como **primera línea** un encabezado con el siguiente formato:


Alumno: Apellido, Nombre - Padrón: NNNNNN

Donde `NNNNNN` es un número de padrón de exactamente 6 dígitos. Por ejemplo:

Alumno: Garcia, Juan - Padrón: 104560

Este encabezado es lo que permite identificar a quién pertenece cada entrega. Si un archivo no lo tiene (o está mal escrito), no se puede corregir y hay que notificarlo.

### Enunciado[](https://www.intro-camejo.com.ar/docs/Material/Clases/bash-complementario#enunciado "Enlace directo al Enunciado")

Escribir un script en Bash llamado `gestionar_entregas.sh` que reciba como primer parámetro una **acción** y como segundo parámetro el **directorio de entregas**. El script va a ser tu herramienta principal para organizar el trabajo de corrección.

Las acciones posibles son:

#### 1\. `inicializar`[](https://www.intro-camejo.com.ar/docs/Material/Clases/bash-complementario#1-inicializar "Enlace directo al 1-inicializar")

$ bash gestionar_entregas.sh inicializar entregas

Debe crear la siguiente estructura de carpetas (solo si no existen):

entregas/  originales/  procesadas/  burlas/

Informar por cada carpeta si fue creada o si ya existía.

#### 2\. `procesar`[](https://www.intro-camejo.com.ar/docs/Material/Clases/bash-complementario#2-procesar "Enlace directo al 2-procesar")

$ bash gestionar_entregas.sh procesar entregas

Debe recorrer todos los archivos `.txt` dentro de `entregas/originales/` y para cada uno:

1.  **Validar el encabezado**: verificar que la primera línea del archivo cumpla con el formato `Alumno: Apellido, Nombre - Padrón: NNNNNN` (donde `NNNNNN` son exactamente 6 dígitos). Si no cumple, mostrar un error para ese archivo y **saltearlo** (no procesarlo).

2.  Generar una versión procesada en `entregas/procesadas/<PADRON>.txt` usando el número de padrón extraído del encabezado (por ejemplo: `104560.txt`). La versión procesada es la que van a leer los correctores, y no necesitan ver el encabezado.

3.  Eliminar la línea del encabezado del archivo procesado (la primera línea con los datos del alumno).

#### 3\. `burlarme`[](https://www.intro-camejo.com.ar/docs/Material/Clases/bash-complementario#3-burlarme "Enlace directo al 3-burlarme")

$ bash gestionar_entregas.sh burlarme entregas

Después de horas de corregir, necesitás un poco de humor. Esta acción recorre todos los archivos `.txt` dentro de `entregas/procesadas/` y para cada uno genera una versión "burlona" en `entregas/burlas/` donde **todas las vocales (a, e, o, u) se reemplazan por la letra `i`**.

Por ejemplo, si `entregas/procesadas/104560.txt` contiene:

La funcion recibe como parametro una lista de enteros.

El archivo `entregas/burlas/104560.txt` debería contener:

Li fincion ricibi cimi pirimitri ini listi di intiris.

Debe reemplazar tanto vocales minúsculas como mayúsculas (las mayúsculas se reemplazan por `I`).

#### 4\. Acción inválida[](https://www.intro-camejo.com.ar/docs/Material/Clases/bash-complementario#4-acción-inválida "Enlace directo al 4. Acción inválida")

Si la acción recibida no es ninguna de las anteriores, mostrar un mensaje de error con las opciones válidas.

### Validaciones generales[](https://www.intro-camejo.com.ar/docs/Material/Clases/bash-complementario#validaciones-generales "Enlace directo al Validaciones generales")

-   Si no se pasan los 2 parámetros necesarios, mostrar un mensaje de uso: `Uso: bash gestionar_entregas.sh <accion> <directorio>`.
-   Si el directorio no existe y la acción no es `inicializar`, mostrar un error.

---

En Bash, las variables definidas dentro de una función son, por defecto, globales.

Si queremos que una variable sea local a la función, usamos `local` antes de su nombre:

```bash
inicializar() {
  local directorio_padre=$1
  ...
}
```

Ejemplo: recorrer y preparar las carpetas necesarias con un `for`:

```bash
for directorio in "$directorio_padre/originales" "$directorio_padre/procesadas" "$directorio_padre/burlas"; do
  ...
done
```

Este `for` itera las tres rutas:
- `$directorio_padre/originales`
- `$directorio_padre/procesadas`
- `$directorio_padre/burlas`

Dentro del for comprobamos si cada ruta existe con `if [ -d "$directorio" ]`:
- `-d`: verifica existencia de directorios
- `-f`: verifica existencia de archivos
- `-e`: verifica existencia de cualquier entrada (archivo o directorio)


La función `inicializar` debe:

1. Guardar el directorio padre en una variable local.
2. Comprobar si existen los subdirectorios `originales`, `procesadas` y `burlas`.
3. Crear los que falten con `mkdir -p` y notificar si se crearon o ya existían.

Nota: `mkdir -p` no falla si el directorio ya existe, gracias a `-p`

La función `procesar` debe realizar los siguientes pasos:

1. Guardar en una variable local el directorio base recibido como parámetro.
2. Definir un `regex` local para validar el encabezado: `^Alumno: [A-Za-z]+, [A-Za-z]+ - Padron: [0-9]{6}$`.
   - `^`: inicio de línea; `$`: fin de línea
   - `[A-Za-z]+`: una o más letras
   - `[0-9]{6}`: exactamente seis dígitos

3. Recorrer todos los archivos `.txt` en `"$directorio/originales"` con: `for archivo in "\$directorio/originales"/*.txt; do`.
4. Para cada archivo, leer la primera línea y el nombre base:

```bash
local primera_linea=$(head -1 "$archivo")
local nombre_archivo=$(basename "$archivo")
```

5. Validar la primera línea contra el `regex` con `if echo "$primera_linea" | grep -qE "$regex"; then`.
   - Si no coincide: informar `El archivo $nombre_archivo no cumple el enunciado.` y usar `continue` para pasar al siguiente archivo.
   - `grep -qE`: `-E` usa expresiones regulares extendidas; `-q` silencia la salida, solo devuelve el código de estado.

6. Si la línea es válida: extraer el padrón con `grep -oE "[0-9]{6}"` y copiar el archivo a `procesadas/${padron}.txt`.
   - Luego eliminar la línea de encabezado del archivo copiado (por ejemplo con `sed -E -i "/$regex/d"`).
   - Informar `Procesamos correctamente el archivo ${nombre_archivo}`.


Función `burlarme`:

Toma los archivos ya procesados y genera una versión "burlada" en la carpeta `burlas` reemplazando las vocales por `i` (minúsculas por `i`, mayúsculas por `I`).

Pasos principales:
1. Recibir el directorio base como parámetro (`local directorio=$1`).
2. Recorrer `"$directorio/procesadas"/*.txt` y obtener el nombre con `basename`.
3. Aplicar las sustituciones con `sed`: `s/[aeou]/i/g; s/[AEOU]/I/g` y escribir el resultado en `"$directorio/burlas/$nombre"`.

Notas:
- Usamos `>` para crear un nuevo archivo y no modificar el original.
- `-i` en `sed` modifica el archivo in-place. Es decir, es destructivo. Por eso no lo usamos.
- El sufijo `g` en `s/patrón/reemplazo/g` indica sustitución global por línea (todas las coincidencias en cada línea).

Ejemplo de `sed`:
```bash
gsed 's/[aeou]/i/g; s/[AEOU]/I/g' "$archivo" > "$directorio/burlas/$nombre"
```


Control de acciones con `case`:

Usamos `case` para ejecutar la función correspondiente según el valor de `$ACCION`:

```bash
case $ACCION in
  inicializar) inicializar "$DIRECTORIO" ;;
  procesar) procesar "$DIRECTORIO" ;;
  burlarme) burlarme "$DIRECTORIO" ;;
  *)
    echo "Error: la acción $ACCION no es conocida"
    exit 1
    ;;
esac
```

Notas rápidas:
- `;;` finaliza cada caso (como un `break`).
- `*)` es el caso por defecto (entrada inválida).
- Usar `exit 1` para errores y `exit 0` para indicar éxito.

Así, según la acción pasada por parámetro se invoca `inicializar`, `procesar` o `burlarme`; si la acción no es válida, el script informa y retorna error.


In [ ]:
#!/bin/bash

ACCION=$1
DIRECTORIO=$2

inicializar() {
  local directorio_padre=$1
  for directorio in "$directorio_padre/originales" "$directorio_padre/procesadas" "$directorio_padre/burlas"; do
    if [ -d "$directorio" ]; then
      echo "El directorio $directorio ya existe"
    else
      mkdir -p "$directorio"
      echo "Se creo correctamente $directorio"
    fi
  done
}

procesar() {
  local directorio=$1
  local regex="^Alumno: [A-Za-z]+, [A-Za-z]+ - Padron: [0-9]{6}$"

  for archivo in "$directorio/originales"/*.txt; do
    local primera_linea=$(head -1 "$archivo")
    local nombre_archivo=$(basename "$archivo")
    if echo "$primera_linea" | grep -qE "$regex"; then
      echo "El archivo $nombre_archivo , esta ok"
    else
      echo "El archivo $nombre_archivo, no cumple el enunciado."
      continue
    fi
    local padron=$(echo "$primera_linea" | grep -oE "[0-9]{6}")
    cp "$archivo" "$directorio/procesadas/${padron}.txt"
    gsed -E -i "/$regex/d" "$directorio/procesadas/${padron}.txt"
    echo "Procesamos correctamente el archivo ${nombre_archivo}"
  done
}


burlarme() {
    local directorio=$1

    for archivo in "$directorio/procesadas"/*.txt; do
        local nombre=$(basename "$archivo")

        gsed 's/[aeou]/i/g; s/[AEOU]/I/g' "$archivo" > "$directorio/burlas/$nombre"
        echo "Burla generada: $directorio/burlas/$nombre"
    done
}

case $ACCION in
  inicializar) inicializar "$DIRECTORIO";;
  procesar) procesar "$DIRECTORIO";;
  burlarme) burlarme "$DIRECTORIO";;
  *)
    echo "Error: la acción $ACCION no es conocida"
    echo "Las validas son inicializar, procesar, burlarme"
    echo "vuelva a intentar"
    exit 1
    ;;
esac
exit 0